In [ ]:
import pandas as pd
import altair as alt
import numpy as np
import openpyxl as px
import os

In [ ]:
df = pd.read_csv('merged_data/nih_merged2_with_traits.csv')

In [ ]:
# converting to datetime for pandas
df['project_start'] = pd.to_datetime(df['project_start'])
df['project_end'] = pd.to_datetime(df['project_end'])
df['first_termination_date'] = pd.to_datetime(df['first_termination_date'])

# verifying the conversion worked
print(df[['project_start', 'project_end', 
          'first_termination_date']].dtypes)
print()
print(df[['project_start', 'project_end', 
          'first_termination_date']].head(5))

In [ ]:
# 
terminated = df[df['current_status'] == 'Disrupted'].copy()

# total project length in days
terminated['total_duration'] = (
    terminated['project_end'] - terminated['project_start']
).dt.days

#time elapsed in days from project start to termination
terminated['days_elapsed'] = (
    terminated['first_termination_date'] - terminated['project_start']
).dt.days

# percentage through the project when terminated
# clip at 0 and 1 to handle any data oddities 
# (a termination date before start or after end)
terminated['pct_elapsed'] = (
    terminated['days_elapsed'] / terminated['total_duration']
).clip(0, 1)

# sanity check before looking at results
print("Null counts in new columns:")
print(terminated[['total_duration', 'days_elapsed', 
                   'pct_elapsed']].isnull().sum())
print()
print(terminated[['total_duration', 'days_elapsed', 
                   'pct_elapsed']].describe().round(2))

In [ ]:
print(df.columns.tolist())

# summarizing the financial waste for disrupted grants
# using the 1234 grants with complete pct_elapsed data
clean_terminated = terminated[terminated['pct_elapsed'].notna()].copy()

print(f"Grants with complete data: {len(clean_terminated)}")
print()
print("SUNK COST (already spent)")
print(f"Total already outlaid: ${clean_terminated['award_outlaid'].sum():,.0f}")
print(f"Median per grant:      ${clean_terminated['award_outlaid'].median():,.0f}")
print()
print("REMAINING (lost potential)")
print(f"Total award remaining: ${clean_terminated['award_remaining'].sum():,.0f}")
print(f"Median per grant:      ${clean_terminated['award_remaining'].median():,.0f}")
print()
print("TOTAL AWARD VALUE")
print(f"Total value of cut grants: ${clean_terminated['award_value'].sum():,.0f}")
print()

# What fraction was already spent?
clean_terminated['pct_spent'] = (
    clean_terminated['award_outlaid'] / clean_terminated['award_value']
)
print("PERCENT SPENT WHEN CUT")
print(f"Median % already spent: {clean_terminated['pct_spent'].median():.1%}")
print(f"Mean % already spent:   {clean_terminated['pct_spent'].mean():.1%}")

In [ ]:
total_value = clean_terminated['award_value'].sum()
sunk = clean_terminated['award_outlaid'].sum()
remaining = clean_terminated['award_remaining'].sum()
clawed_back = terminated['post_termination_deobligation'].sum()

print(f"Total award value committed:     ${total_value:,.0f}")
print(f"Already spent (irrecoverable):   ${sunk:,.0f}")
print(f"Committed but unspent:           ${remaining:,.0f}")
print(f"Clawed back after termination:   ${abs(clawed_back):,.0f}")
print(f"Net unrecovered remaining:       ${remaining - abs(clawed_back):,.0f}")
print()
print(f"Government recovered:            {abs(clawed_back)/total_value:.1%} of total")
print(f"Irrecoverable sunk cost:         {sunk/total_value:.1%} of total")

In [ ]:


# Chart 1. Distribution of how far along grants were when terminated

median_val = terminated['pct_elapsed'].median()

# histogram bars
histogram = alt.Chart(terminated[['pct_elapsed']]).mark_bar(  
    color='steelblue',
    opacity=0.7


).encode(
    # Note for self:bin=True lets Altair choose bin sizes automatically
    x=alt.X('pct_elapsed:Q',
            bin=alt.Bin(step=0.05),
            title='Fraction of Project Period Elapsed at Termination',
            axis=alt.Axis(format='%')),   # formatting 0.61 as 61%
    y=alt.Y('count()',
            title='Number of Disrupted Grants without Restoration')
)

# A vertical rule (line) at the median
# put the median value into a tiny dataframe because Altair needs a dataframe to plot from remember this later
median_line = alt.Chart(
    pd.DataFrame({'median': [median_val]})
).mark_rule(
    color='firebrick',
    strokeWidth=2,
    strokeDash=[6, 3]   # dashed line: 6px on, 3px off
).encode(
    x='median:Q'
)

# adding a text label sitting on the line
median_label = alt.Chart(
    pd.DataFrame({'median': [median_val], 'label': [f'Median: {median_val:.0%}']})
).mark_text(
    align='left',
    dx=6,           # moving the text 6px to the right of the line
    dy=-80,         # moving it up so it doesn't overlap bars
    color='firebrick',
    fontSize=12
).encode(
    x='median:Q',
    text='label:N'
)

# layering the three pieces together with +
chart1 = (histogram + median_line + median_label).properties(
    title='Terminated Grants Were Typically 60% Through Their Project Period',
    width=500,
    height=300
)

chart1

In [ ]:
# defining variables because altair defaults to using giga-billions?
total_sunk     = clean_terminated['award_outlaid'].sum()
total_frozen   = clean_terminated['award_remaining'].sum()
deob           = clean_terminated['post_termination_deobligation'].abs().sum()
in_limbo       = total_frozen - deob
total_combined = total_sunk + total_frozen
n_grants       = clean_terminated['award_outlaid'].notna().sum()

order = ['Spent Before Termination (Irrecoverable)',
         'Committed but Unspent (In Limbo)',
         'Formally Deobligated (Recovered)']

money_data = pd.DataFrame({
    'Category':   ['Spent Before Termination (Irrecoverable)',
                   'Committed but Unspent (In Limbo)',
                   'Formally Deobligated (Recovered)'],
    'Amount_M':   [total_sunk/1e6, in_limbo/1e6, deob/1e6],
    'bar':        ['All disrupted grants'] * 3,
    'sort_order': [0, 1, 2],
})

chart2 = alt.Chart(money_data).mark_bar().encode(
    x=alt.X('Amount_M:Q',
            title='Millions of dollars ($ Millions)',
            axis=alt.Axis(format='$,.0f'),
            stack='zero'),
    y=alt.Y('bar:N', title=None, axis=alt.Axis(labels=False, ticks=False)),
    color=alt.Color('Category:N',
        scale=alt.Scale(domain=order,
                        range=['#1f77b4',
                               '#258b25',
                               '#9467bd']),
        sort=order,
        legend=alt.Legend(
            title=None,
            orient='bottom',
            columns=3,
            labelLimit=220,
        )),
    order=alt.Order('sort_order:Q', sort='ascending'),
    tooltip=[alt.Tooltip('Category:N', title='Fate of the dollar'),
             alt.Tooltip('Amount_M:Q', format='$,.1f', title='Amount ($ Millions)')],
).properties(
    width=560, height=120,
    title=alt.TitleParams(
        text=f'${total_combined/1e9:.2f}B committed across {n_grants:,} disrupted grants',
        subtitle='801 grants with complete spending data; hover any segment for its share'
    )
)
chart2

In [ ]:
# Building the two frames the scatter and heatmap 
base = terminated.dropna(subset=['years_funded', 'award_outlaid']).copy()

scatter_df = base.copy()   # Chart 3: raw scatter, keeps outliers (honest)
heat_df = base[(base['award_outlaid'] > 0) &
               (base['award_outlaid'] < 20_000_000)].copy()   # Chart 4: log heatmap

# reporting exactly what the heatmap excludes, sourced from `terminated`
n_zero = (terminated['award_outlaid'] == 0).sum()
n_neg  = (terminated['award_outlaid'] < 0).sum()
n_big  = (terminated['award_outlaid'] >= 20_000_000).sum()
print(f"Excluded from log heatmap → zeros: {n_zero}, negatives: {n_neg}, >=$20M: {n_big}")

In [ ]:
# Chart 3. Time invested vs. money already spent

areas = sorted(scatter_df['funding_category'].dropna().unique().tolist())

area_dropdown = alt.binding_select(
    options=[None] + areas,
    labels=['All areas'] + areas,
    name='Research area: '
)
area_sel = alt.selection_point(fields=['funding_category'], bind=area_dropdown)

# color map — domain strings MUST match your data exactly (see the print above)
color_scale = alt.Scale(
    domain=['Other Transactions', 'Research and Training',
            'Research and Development', 'Small Business'],
    range=['#999999',   # grey Other Transactions
           '#258b25',   # blue Research and Training
           "#1f77b4",   # green Research and Development
           '#9467bd'],  # purple Small Business
)

chart3 = (
    alt.Chart(scatter_df[['years_funded', 'award_outlaid', 'funding_category', 'title']])
    .mark_circle(opacity=0.55)
    .encode(
        x=alt.X('years_funded:Q',
                title='Total fiscal years of NIH funding before termination',
                axis=alt.Axis(format='d')),
        y=alt.Y('award_outlaid:Q', title='Money already spent ($)',
                axis=alt.Axis(format='$,.0f')),
        color=alt.Color('funding_category:N', scale=color_scale,
                        title='Research area'),
        tooltip=['title', 'years_funded', 'award_outlaid', 'funding_category'],
    )
    .add_params(area_sel)
    .transform_filter(area_sel)
    .properties(width=600, height=400,
                title='Money already spent, by total years of funding before termination')
    .interactive()
)
chart3

This chart shows 810 of the 1,249 disrupted grants with both a recorded funding length and spending amount.  The remaining 439 are excluded because they lack one of those values, almost all (434) missing a recorded outlay in NIH RePORTER despite being fully terminated.  Because it is unclear whether these grants spent little or simply have incomplete spending records, this view should be read as covering the 810 grants with complete data rather than all disrupted grants.

In [ ]:
print(scatter_df['funding_category'].value_counts(dropna=False))

In [ ]:
#Debugging :( Note to me to hashtag out this line if I want to run this cell

# cheecking what chart is plotting
print("scatter_df total rows:", len(scatter_df))
print("plottable (both axes non-null):",
      scatter_df[['years_funded', 'award_outlaid']].notna().all(axis=1).sum())

# counts per year, and per year-by-category (what color splits into)
print("\nGrants at year 1:", (scatter_df['years_funded'] == 1).sum())
print("Grants at year 2:", (scatter_df['years_funded'] == 2).sum())
# counts per year-by-category (what color splits into)
print("\nYear 1 by category:")
print(scatter_df[scatter_df['years_funded'] == 1]['funding_category'].value_counts(dropna=False))
print("\nYear 2 by category:")
print(scatter_df[scatter_df['years_funded'] == 2]['funding_category'].value_counts(dropna=False))

print("disrupted grants (terminated):", len(terminated))
print("missing years_funded:", terminated['years_funded'].isna().sum())
print("missing award_outlaid:", terminated['award_outlaid'].isna().sum())
print("missing either (dropped by scatter_df):",
      terminated[['years_funded', 'award_outlaid']].isna().any(axis=1).sum())
print("kept in scatter_df:",
      terminated[['years_funded', 'award_outlaid']].notna().all(axis=1).sum())

In [ ]:
# Chart 4. Where terminated grants cluster (binned, log y-axis)
base = terminated.dropna(subset=['years_funded', 'award_outlaid']).copy()
scatter_df = base.copy()   # Chart 3: raw, keeps all points
heat_df = base[(base['award_outlaid'] >= 0) &
               (base['award_outlaid'] < 20_000_000)].copy()   # drops -$141 and the $48M; keeps zeros

print(f"Chart 4 excludes → negatives: {(base['award_outlaid'] < 0).sum()}, "
      f">=$20M: {(base['award_outlaid'] >= 20_000_000).sum()}")

heat = (
    alt.Chart(heat_df[['years_funded', 'award_outlaid']])
    .mark_rect()
    .encode(
        x=alt.X('years_funded:Q', bin=alt.Bin(step=1),
                title='Total fiscal years of NIH funding before termination',
                axis=alt.Axis(format='d')),
        y=alt.Y('award_outlaid:Q', bin=alt.Bin(maxbins=30),
                title='Money already spent ($)',
                axis=alt.Axis(format='$,.0f')),
        color=alt.Color('count():Q', title='Number of grants',
                        scale=alt.Scale(scheme='blues')),
        tooltip=[alt.Tooltip('count():Q', title='Grants')],
    )
    .properties(width=620, height=420,
                title='Where terminated grants cluster: money spent vs. years funded')
)
heat.save('years_vs_spent.html')
heat

In [ ]:
zeros = terminated[terminated['award_outlaid'] == 0][
    ['title', 'award_value', 'award_remaining']
].copy()
print(f"{len(zeros)} grants spent $0 before termination")   # 13
for _, r in zeros.iterrows():
    print(f"<tr><td>{r['title']}</td>"
          f"<td>${r['award_value']:,.0f}</td>"
          f"<td>${r['award_remaining']:,.0f}</td></tr>")

This chart shows 808 of the 1,249 disrupted grants.  It excludes 439 grants missing a recorded outlay or funding length, one grant with a negative recorded outlay (−$141, a deobligation artifact), and one $48.1M award (UTMB–Novartis Alliance for Pandemic Preparedness).  All excluded grants remain in every reported dollar total.

In [ ]:
#Debugging :( Note to me to hashtag out this line if I want to run this cell
%%script false --no-raise-error

#checking counts of disrupted grants, those plotted in scatter_df, and those plotted in heat_df
n_disrupted = len(terminated)                                 
n_base      = len(base)                                        
n_dropped   = n_disrupted - n_base                            
n_missing_outlay = terminated['award_outlaid'].isna().sum()
n_missing_years  = terminated['years_funded'].isna().sum()
n_heat      = len(heat_df)                                     

#confirming scatter_df and heat_df counts
print(f"Disrupted grants:              {n_disrupted}")
print(f"Plotted in scatter (base):     {n_base}")
print(f"Dropped (missing a value):     {n_dropped}")
print(f"-missing award_outlaid:     {n_missing_outlay}")
print(f"-missing years_funded:      {n_missing_years}")
print(f"Plotted in heatmap (heat_df):  {n_heat}")

#checking outlay missing values in the terminated grants
missing = terminated[terminated['award_outlaid'].isna()]
print(missing['detailed_status'].value_counts(dropna=False))
print("\nHow many are frozen vs terminated:")
print(missing['current_status'].value_counts(dropna=False))

#NaN counts in the relevant columns
print("award_outlaid non-null count:", terminated['award_outlaid'].notna().sum())
print("Sum treats missing as:", "skipped (not zero)")  # pandas .sum() skips NaN by default
print("Grants terminated but missing outlay:", 
      ((terminated['current_status']=='Disrupted') & terminated['award_outlaid'].isna()).sum())

#Checking the number of grants with recorded outlay and the total sunk cost
print("Grants with recorded outlay:", terminated['award_outlaid'].notna().sum())      # 815
print("Sunk cost (all terminated w/ outlay): $%s" % f"{terminated['award_outlaid'].sum():,.0f}")
print("Sunk cost (clean_terminated):         $%s" % f"{clean_terminated['award_outlaid'].sum():,.0f}")
print("Grants w/ outlay in clean_terminated:", clean_terminated['award_outlaid'].notna().sum())

#Checking for missing values in the relevant columns
print("award_outlaid non-null:  ", terminated['award_outlaid'].notna().sum())
print("award_remaining non-null:", terminated['award_remaining'].notna().sum())
print("both non-null:           ", terminated[['award_outlaid','award_remaining']].notna().all(axis=1).sum())
print("award_value non-null:    ", terminated['award_value'].notna().sum())

In [ ]:
# os.makedirs('static', exist_ok=True)

# chart1.save('static/chart1.png')
# chart2.save('static/chart2.png')
# chart3.save('static/chart3.png')
# heat.save('static/chart4.png')     # Chart 4's variable is `heat`

# print("saved 4 PNGs to static/")

In [ ]:

os.makedirs('static', exist_ok=True)
chart1.save('static/chart1.html')
chart2.save('static/chart2.html')
chart3.save('static/chart3.html')
heat.save('static/chart4.html')
print("saved charts to static folder/")